# Multilingual NER Model Fine-Tuning

This notebook handles the fine-tuning of the XLM-RoBERTa model using the interleaved Hungarian, English, and German datasets.

## Environment Setup

Initializing hardware settings and loading core libraries for training.

In [21]:
import os
from dotenv import load_dotenv
import torch
import evaluate
import wandb
import numpy as np
from datasets import load_from_disk
from transformers import(
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    pipeline
)
from transformers.utils.notebook import NotebookProgressCallback 
from huggingface_hub import notebook_login

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

load_dotenv()

cuda


True

## Weights & Biases Logging

Connecting to W&B to track metrics during the experiment runs.

In [2]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rchrdhlzhfr (rchrdhlzhfr-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Loading Processed Datasets

Loading the tokenized and aligned datasets.

In [3]:
# Load interleaved and gold-only datasets
interleaved_dataset = load_from_disk("../data/processed/harmonized_dataset/interleaved_dataset")
tokenized_dataset = load_from_disk("../data/processed/tokenized_dataset/tokenized_dataset")

gold_only_dataset = load_from_disk("../data/processed/harmonized_dataset/gold_only_dataset")
gold_only_tokenized_dataset = load_from_disk("../data/processed/tokenized_dataset/gold_only_tokenized_dataset")

# Load individual language datasets (harmonized)
hun_dataset = load_from_disk("../data/processed/harmonized_dataset/hun_dataset")
eng_dataset = load_from_disk("../data/processed/harmonized_dataset/eng_dataset")
ger_dataset = load_from_disk("../data/processed/harmonized_dataset/ger_dataset")

# Load individual language datasets (tokenized)
tokenized_hun_dataset = load_from_disk("../data/processed/tokenized_dataset/hun_tokenized_dataset")
tokenized_eng_dataset = load_from_disk("../data/processed/tokenized_dataset/eng_tokenized_dataset")
tokenized_ger_dataset = load_from_disk("../data/processed/tokenized_dataset/ger_tokenized_dataset")

## Tokenizer, model and data collator initialization

In [4]:
model_id = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

Preparing dictionaries for model initialization on interleaved dataset

In [5]:
label_names = interleaved_dataset['train']['ner'].features.feature.names
label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [6]:
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

In [7]:
model = AutoModelForTokenClassification.from_pretrained(
    model_id,
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Data Collator Verification

Ensuring labels are correctly padded and aligned within batches.

In [8]:
# Lengths of the first three labels before data collator
[print(len(tokenized_dataset['train']['labels'][i])) for i in range(3)];

60
87
20


In [9]:
example_batch = data_collator([tokenized_dataset["train"][i] for i in range(3)])
example_batch["labels"]

tensor([[-100,    0,    7, -100, -100,    8, -100, -100,    0,    0,    0, -100,
            0,    7, -100,    0,    0,    0, -100,    0, -100,    0,    0, -100,
            0, -100, -100,    0,    0,    0, -100,    0,    0, -100, -100,    0,
            0, -100, -100,    0,    0, -100,    0, -100,    0,    0, -100, -100,
         -100,    0,    0, -100, -100,    0,    0, -100, -100,    0, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100],
        [-100,    1, -100, -100,    0,    0, -100, -100, -100, -100, -100, -100,
            0,    0, -100, -100, -100, -100, -100, -100,    0,    0,    0,    0,
            0,    0,    0,    0,    0, -100,    0,    0, -100,    0,    0,    0,
            0,    0,    0, -100, -100, -100, -100,    0, -100,    0,    0,    0,
            0,    0,    0,    0, -100,    0,    0, -100,    0, -100,    0, -100,


Inspecting the first three labels after data collator

In [10]:
# Lengths of the first three labels after data collator
[print(len(example_batch['labels'][i])) for i in range(3)];

87
87
87


## Evaluation Metrics

Defining the `compute_metrics` function using `seqeval` to calculate Precision, Recall, and F1 score for all entity tags.

In [11]:
metric = evaluate.load("seqeval")

In [13]:
def compute_metrics(eval_preds, label_names):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    results = {
        "overall_precision": all_metrics["overall_precision"],
        "overall_recall": all_metrics["overall_recall"],
        "overall_f1": all_metrics["overall_f1"],
        "overall_accuracy": all_metrics["overall_accuracy"],
    }

    for k, v in all_metrics.items():
        if k not in ["overall_precision", "overall_recall", "overall_f1", "overall_accuracy"]:
            results[f"{k}_f1"] = v["f1"]
            results[f"{k}_precision"] = v["precision"]
            results[f"{k}_recall"] = v["recall"]
    
    return results           
    

## Training Configuration

Setting hyper-parameters such as learning rate, batch size, and weight decay.

In [14]:
# Login to HuggingFace
notebook_login()

In [15]:
args = TrainingArguments(
    output_dir="../xlm-roberta-ner-hun_eng_ger",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    metric_for_best_model="eval_combined_loss",
    load_best_model_at_end=True,
    greater_is_better=False,
    per_device_train_batch_size=96,
    per_device_eval_batch_size=192,
    bf16=True,
    bf16_full_eval=True,
    dataloader_num_workers=8,
    dataloader_persistent_workers=True,
    dataloader_prefetch_factor=2,
    learning_rate=1e-5,
    num_train_epochs=5,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    warmup_ratio=0.1,
    push_to_hub=False,
    report_to="wandb",
    run_name="ner_exp_hun_eng_ger"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## Trainer Setup & Training

Starting the training process with the Hugging Face `Trainer`.

In [16]:
eval_datasets = {
    "combined": tokenized_dataset["validation"],
    "hun": tokenized_hun_dataset["validation"],
    "eng": tokenized_eng_dataset["validation"],
    "ger": tokenized_ger_dataset["validation"]
}

In [17]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=eval_datasets,
    data_collator=data_collator,
    compute_metrics=lambda x: compute_metrics(x, label_names=label_names),
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [18]:
wandb.init(
    entity="rchrdhlzhfr-personal",
    project="polyglot-ner-project",
    name="hun_eng_ger_training"
    )

trainer.train()

wandb.finish()

Epoch,Training Loss,Validation Loss,Combined Loss,Combined Overall Precision,Combined Overall Recall,Combined Overall F1,Combined Overall Accuracy,Combined Loc F1,Combined Loc Precision,Combined Loc Recall,Combined Misc F1,Combined Misc Precision,Combined Misc Recall,Combined Org F1,Combined Org Precision,Combined Org Recall,Combined Per F1,Combined Per Precision,Combined Per Recall,Hun Loss,Hun Overall Precision,Hun Overall Recall,Hun Overall F1,Hun Overall Accuracy,Hun Loc F1,Hun Loc Precision,Hun Loc Recall,Hun Misc F1,Hun Misc Precision,Hun Misc Recall,Hun Org F1,Hun Org Precision,Hun Org Recall,Hun Per F1,Hun Per Precision,Hun Per Recall,Eng Loss,Eng Overall Precision,Eng Overall Recall,Eng Overall F1,Eng Overall Accuracy,Eng Loc F1,Eng Loc Precision,Eng Loc Recall,Eng Misc F1,Eng Misc Precision,Eng Misc Recall,Eng Org F1,Eng Org Precision,Eng Org Recall,Eng Per F1,Eng Per Precision,Eng Per Recall,Ger Loss,Ger Overall Precision,Ger Overall Recall,Ger Overall F1,Ger Overall Accuracy,Ger Loc F1,Ger Loc Precision,Ger Loc Recall,Ger Misc F1,Ger Misc Precision,Ger Misc Recall,Ger Org F1,Ger Org Precision,Ger Org Recall,Ger Per F1,Ger Per Precision,Ger Per Recall
1,0.590205,No log,0.065642,0.745135,0.824878,0.782982,0.971967,0.741291,0.703704,0.783121,0.643735,0.719780,0.582222,0.795838,0.779454,0.812925,0.813896,0.749837,0.889922,0.019824,0.935595,0.947709,0.941613,0.994455,0.938567,0.910596,0.968310,0.782090,0.891156,0.696809,0.964399,0.952344,0.976763,0.937605,0.911475,0.965278,0.136671,0.537768,0.671516,0.597245,0.947944,0.415157,0.375000,0.464945,0.000000,0.000000,0.000000,0.278689,0.266458,0.292096,0.750343,0.661027,0.867565,0.061328,0.796449,0.851440,0.823027,0.981444,0.854516,0.828682,0.882013,0.000000,0.000000,0.000000,0.730216,0.712281,0.749077,0.866385,0.838475,0.896217
2,0.084396,No log,0.055232,0.800109,0.818346,0.809125,0.975450,0.764313,0.743733,0.786065,0.740047,0.782178,0.702222,0.824209,0.829174,0.819303,0.837041,0.821575,0.853101,0.012094,0.958292,0.961155,0.959722,0.996521,0.946274,0.931741,0.961268,0.855586,0.877095,0.835106,0.977227,0.974502,0.979968,0.963478,0.965157,0.961806,0.123599,0.601826,0.618489,0.610044,0.952647,0.432618,0.404494,0.464945,0.041667,0.090909,0.027027,0.239175,0.298969,0.199313,0.767823,0.739354,0.798573,0.050694,0.837984,0.864856,0.851208,0.984651,0.879641,0.870056,0.889439,0.000000,0.000000,0.000000,0.767497,0.741123,0.795818,0.890830,0.891262,0.890398
3,0.073585,No log,0.053351,0.809543,0.825295,0.817343,0.976131,0.767084,0.740301,0.795878,0.761905,0.820513,0.711111,0.831824,0.843832,0.820153,0.849514,0.836275,0.863178,0.011248,0.959305,0.962649,0.960974,0.996439,0.948276,0.929054,0.968310,0.854054,0.868132,0.840426,0.977582,0.976800,0.978365,0.970435,0.972125,0.968750,0.126107,0.612029,0.625528,0.618705,0.953529,0.437500,0.403427,0.477860,0.081633,0.166667,0.054054,0.220807,0.288889,0.178694,0.783109,0.758929,0.808882,0.046438,0.850414,0.874346,0.862214,0.985558,0.882759,0.868316,0.897690,0.000000,0.000000,0.000000,0.786099,0.766355,0.806888,0.899661,0.899225,0.900097
4,0.062028,No log,0.056326,0.799519,0.830855,0.814885,0.974594,0.764720,0.727313,0.806183,0.739910,0.746606,0.733333,0.833511,0.837989,0.829082,0.845714,0.831461,0.860465,0.011348,0.955512,0.962649,0.959067,0.996548,0.941980,0.913907,0.971831,0.851852,0.847368,0.856383,0.977154,0.977546,0.976763,0.968531,0.975352,0.961806,0.136848,0.595103,0.638667,0.616116,0.949587,0.442260,0.397644,0.498155,0.131148,0.166667,0.108108,0.284600,0.328829,0.250859,0.773161,0.744493,0.804124,0.047871,0.846639,0.877945,0.862008,0.985306,0.881124,0.858372,0.905116,0.000000,0.000000,0.000000,0.785415,0.763953,0.808118,0.903696,0.906341,0.901067
5,0.059374,No log,0.056919,0.796831,0.831828,0.813953,0.974568,0.765340,0.732079,0.801766,0.736383,0.722222,0.751111,0.832269,0.833333,0.831207,0.843720,0.825120,0.863178,0.011174,0.958025,0.966135,0.962063,0.996738,0.948454,0.926174,0.971831,0.858639,0.845361,0.872340,0.978775,0.978383,0.979167,0.

early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
d:\Projects\polyglot_ner_project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
d:\Projects\polyglot_ner_project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
d:\Projects\polyglot_ner_project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

eval/combined_LOC_f1,▁▇█▇█
eval/combined_LOC_precision,▁█▇▅▆
eval/combined_LOC_recall,▁▂▅█▇
eval/combined_MISC_f1,▁▇█▇▆
eval/combined_MISC_precision,▁▅█▃▁
eval/combined_MISC_recall,▁▆▆▇█
eval/combined_ORG_f1,▁▆███
eval/combined_ORG_precision,▁▆█▇▇
eval/combined_ORG_recall,▁▃▄▇█
eval/combined_PER_f1,▁▆█▇▇
+75,...


## Final Test Set Evaluation

Performing a comprehensive evaluation across separate Hungarian, English, and German test sets to assess cross-lingual generalization.

In [19]:
test_datasets = {
    "combined": tokenized_dataset["test"],
    "test_hun": tokenized_hun_dataset["test"],
    "test_eng": tokenized_eng_dataset["test"],
    "test_ger": tokenized_ger_dataset["test"]
}

In [22]:
# Disabling all multiprocessing features for stable evaluation
trainer.args.dataloader_num_workers = 0 
trainer.args.dataloader_persistent_workers = False
trainer.args.dataloader_prefetch_factor = None

# Fix 'on_train_begin' error by removing notebook callback
trainer.remove_callback(NotebookProgressCallback)

wandb.init(
    entity="rchrdhlzhfr-personal",
    project="polyglot-ner-project",
    name="hun_eng_ger_test"
    )
trainer.args.dataloader_num_workers = 0 
trainer.args.dataloader_persistent_workers = False
trainer.args.dataloader_prefetch_factor = None

test_results = trainer.evaluate(eval_dataset=test_datasets)

wandb.finish()


early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
d:\Projects\polyglot_ner_project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled


eval/combined_LOC_f1,▁
eval/combined_LOC_precision,▁
eval/combined_LOC_recall,▁
eval/combined_MISC_f1,▁
eval/combined_MISC_precision,▁
eval/combined_MISC_recall,▁
eval/combined_ORG_f1,▁
eval/combined_ORG_precision,▁
eval/combined_ORG_recall,▁
eval/combined_PER_f1,▁
+72,...


#### Inspecting testing results

Charts in Weights&Biases are clearly showing that the model performs significantly worse on the English dataset in all types of labels.

In [23]:
test_results.keys()

dict_keys(['eval_combined_loss', 'eval_combined_overall_precision', 'eval_combined_overall_recall', 'eval_combined_overall_f1', 'eval_combined_overall_accuracy', 'eval_combined_LOC_f1', 'eval_combined_LOC_precision', 'eval_combined_LOC_recall', 'eval_combined_MISC_f1', 'eval_combined_MISC_precision', 'eval_combined_MISC_recall', 'eval_combined_ORG_f1', 'eval_combined_ORG_precision', 'eval_combined_ORG_recall', 'eval_combined_PER_f1', 'eval_combined_PER_precision', 'eval_combined_PER_recall', 'eval_combined_runtime', 'eval_combined_samples_per_second', 'eval_combined_steps_per_second', 'epoch', 'eval_test_hun_loss', 'eval_test_hun_overall_precision', 'eval_test_hun_overall_recall', 'eval_test_hun_overall_f1', 'eval_test_hun_overall_accuracy', 'eval_test_hun_LOC_f1', 'eval_test_hun_LOC_precision', 'eval_test_hun_LOC_recall', 'eval_test_hun_MISC_f1', 'eval_test_hun_MISC_precision', 'eval_test_hun_MISC_recall', 'eval_test_hun_ORG_f1', 'eval_test_hun_ORG_precision', 'eval_test_hun_ORG_recal

In [24]:
# Most important metrics
print(f"Overall F1 score:               {test_results["eval_combined_overall_f1"]*100:.2f}%")
print(f"F1 score on Hungarian dataset:  {test_results["eval_test_hun_overall_f1"]*100:.2f}%")
print(f"F1 score on English dataset:    {test_results["eval_test_eng_overall_f1"]*100:.2f}%")
print(f"F1 score on German dataset:     {test_results["eval_test_ger_overall_f1"]*100:.2f}%")

Overall F1 score:               80.51%
F1 score on Hungarian dataset:  96.29%
F1 score on English dataset:    59.44%
F1 score on German dataset:     85.27%


Labels per category in the English dataset

In [25]:
print(f"F1 score on PER labels:     {test_results["eval_test_eng_PER_f1"]*100:.2f}%")
print(f"F1 score on LOC labels:     {test_results["eval_test_eng_LOC_f1"]*100:.2f}%")
print(f"F1 score on ORG labels:     {test_results["eval_test_eng_ORG_f1"]*100:.2f}%")
print(f"F1 score on MISC labels:    {test_results["eval_test_eng_MISC_f1"]*100:.2f}%")


F1 score on PER labels:     74.81%
F1 score on LOC labels:     47.07%
F1 score on ORG labels:     23.33%
F1 score on MISC labels:    0.00%


Since the English dataset performance on all labels is weak, we might suspect that it acts as noise in the model. This is could be attributed to its quality (silver).

The performance on the English dataset is so weak, that is doesn't worth spending time on other possible label-mappings, we should rather focus on defining clear boundaries for Hungarian and German language for the model by training it on the other two datasets.

## Fine tuning on Gold-only dataset

Creating labels from gold_only dataset for compute metrics function

In [26]:
gold_only_label_names = gold_only_dataset['train']['ner'].features.feature.names


In [27]:
gold_only_label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [28]:
args = TrainingArguments(
    output_dir="../xlm-roberta-ner-hun_ger",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    metric_for_best_model="eval_combined_loss",
    load_best_model_at_end=True,
    greater_is_better=False,
    per_device_train_batch_size=96,
    per_device_eval_batch_size=192,
    bf16=True,
    bf16_full_eval=True,
    dataloader_num_workers=8,
    dataloader_persistent_workers=True,
    dataloader_prefetch_factor=2,
    learning_rate=1e-5,
    num_train_epochs=5,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    warmup_ratio=0.1,
    push_to_hub=False,
    report_to="wandb",
    run_name="ner_exp_hun_ger"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [29]:
eval_datasets = {
    "combined": gold_only_tokenized_dataset["validation"],
    "hun": tokenized_hun_dataset["validation"],
    "ger": tokenized_ger_dataset["validation"]
}

In [30]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=gold_only_tokenized_dataset["train"],
    eval_dataset=eval_datasets,
    data_collator=data_collator,
    compute_metrics=lambda x: compute_metrics(x, label_names=gold_only_label_names),
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [31]:
wandb.init(
    entity="rchrdhlzhfr-personal",
    project="polyglot-ner-project",
    name="hun_ger_training"
    )

trainer.train()

wandb.finish()

Epoch,Training Loss,Validation Loss,Combined Loss,Combined Overall Precision,Combined Overall Recall,Combined Overall F1,Combined Overall Accuracy,Combined Loc F1,Combined Loc Precision,Combined Loc Recall,Combined Misc F1,Combined Misc Precision,Combined Misc Recall,Combined Org F1,Combined Org Precision,Combined Org Recall,Combined Per F1,Combined Per Precision,Combined Per Recall,Hun Loss,Hun Overall Precision,Hun Overall Recall,Hun Overall F1,Hun Overall Accuracy,Hun Loc F1,Hun Loc Precision,Hun Loc Recall,Hun Misc F1,Hun Misc Precision,Hun Misc Recall,Hun Org F1,Hun Org Precision,Hun Org Recall,Hun Per F1,Hun Per Precision,Hun Per Recall,Ger Loss,Ger Overall Precision,Ger Overall Recall,Ger Overall F1,Ger Overall Accuracy,Ger Loc F1,Ger Loc Precision,Ger Loc Recall,Ger Misc F1,Ger Misc Precision,Ger Misc Recall,Ger Org F1,Ger Org Precision,Ger Org Recall,Ger Per F1,Ger Per Precision,Ger Per Recall
1,0.021942,No log,0.036209,0.886381,0.915087,0.900505,0.989516,0.894977,0.873885,0.917112,0.852713,0.829146,0.877660,0.900643,0.884831,0.917031,0.913702,0.912320,0.915087,0.011269,0.958436,0.964641,0.961529,0.996602,0.948454,0.926174,0.971831,0.866142,0.854922,0.877660,0.977136,0.978313,0.975962,0.970332,0.975439,0.965278,0.048331,0.841186,0.882199,0.861204,0.985155,0.882448,0.861635,0.904290,0.000000,0.000000,0.000000,0.789196,0.755056,0.826568,0.897485,0.894889,0.900097
2,0.021953,No log,0.035519,0.887737,0.913507,0.900438,0.989703,0.893714,0.867296,0.921791,0.840206,0.815000,0.867021,0.900288,0.889626,0.911208,0.917459,0.920611,0.914329,0.011199,0.955534,0.963147,0.959325,0.996439,0.946827,0.923077,0.971831,0.853403,0.840206,0.867021,0.975942,0.976726,0.975160,0.970332,0.975439,0.965278,0.047348,0.845309,0.881545,0.863047,0.985591,0.881695,0.855039,0.910066,0.000000,0.000000,0.000000,0.789756,0.765589,0.815498,0.902724,0.905366,0.900097
3,0.021465,No log,0.035269,0.890685,0.913902,0.902144,0.989796,0.896597,0.878205,0.915775,0.844560,0.823232,0.867021,0.901745,0.888784,0.915090,0.917647,0.918693,0.916603,0.010925,0.957426,0.963147,0.960278,0.996493,0.950086,0.929293,0.971831,0.855643,0.844560,0.867021,0.976334,0.977510,0.975160,0.970332,0.975439,0.965278,0.047100,0.847655,0.881217,0.864110,0.985642,0.884040,0.866192,0.902640,0.000000,0.000000,0.000000,0.790533,0.761688,0.821648,0.902569,0.902132,0.903007
4,0.021066,No log,0.035258,0.890664,0.913705,0.902037,0.989786,0.895425,0.875959,0.915775,0.846753,0.827411,0.867021,0.902346,0.890411,0.914605,0.917299,0.917995,0.916603,0.011030,0.956952,0.963147,0.960040,0.996521,0.948454,0.926174,0.971831,0.857895,0.848958,0.867021,0.976334,0.977510,0.975160,0.968641,0.972028,0.965278,0.047035,0.849259,0.881217,0.864943,0.985659,0.883683,0.865506,0.902640,0.000000,0.000000,0.000000,0.793349,0.766935,0.821648,0.903007,0.903007,0.903007
5,0.020924,No log,0.035251,0.890857,0.913902,0.902232,0.989848,0.895356,0.876440,0.915107,0.851948,0.832487,0.872340,0.902392,0.890042,0.915090,0.917299,0.917995,0.916603,0.011011,0.957921,0.963645,0.960775,0.996493,0.948454,0.926174,0.971831,0.863158,0.854167,0.872340,0.976334,0.977510,0.975160,0.970332,0.975439,0.965278,0.047034,0.847922,0.881217,0.864249,0.985709,0.882876,0.864715,0.901815,0.000000,0.000000,0.000000,0.792654,0.764571,0.822878,0.902569,0.902132,0.903007


early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
d:\Projects\polyglot_ner_project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
d:\Projects\polyglot_ner_project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
d:\Projects\polyglot_ner_project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

eval/combined_LOC_f1,▄▁█▅▅
eval/combined_LOC_precision,▅▁█▇▇
eval/combined_LOC_recall,▃█▂▂▁
eval/combined_MISC_f1,█▁▃▅█
eval/combined_MISC_precision,▇▁▄▆█
eval/combined_MISC_recall,█▁▁▁▅
eval/combined_ORG_f1,▂▁▆██
eval/combined_ORG_precision,▁▇▆██
eval/combined_ORG_recall,█▁▆▅▆
eval/combined_PER_f1,▁██▇▇
+55,...


In [32]:
test_datasets = {
    "combined": gold_only_tokenized_dataset["test"],
    "test_hun": tokenized_hun_dataset["test"],
    "test_ger": tokenized_ger_dataset["test"]
}

In [33]:
# Disabling all multiprocessing features for stable evaluation
trainer.args.dataloader_num_workers = 0 
trainer.args.dataloader_persistent_workers = False
trainer.args.dataloader_prefetch_factor = None

# Fix 'on_train_begin' error by removing notebook callback
trainer.remove_callback(NotebookProgressCallback)

wandb.init(
    entity="rchrdhlzhfr-personal",
    project="polyglot-ner-project",
    name="hun_ger_test"
    )
trainer.args.dataloader_num_workers = 0 
trainer.args.dataloader_persistent_workers = False
trainer.args.dataloader_prefetch_factor = None

test_results = trainer.evaluate(eval_dataset=test_datasets)

wandb.finish()


early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
d:\Projects\polyglot_ner_project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled


eval/combined_LOC_f1,▁
eval/combined_LOC_precision,▁
eval/combined_LOC_recall,▁
eval/combined_MISC_f1,▁
eval/combined_MISC_precision,▁
eval/combined_MISC_recall,▁
eval/combined_ORG_f1,▁
eval/combined_ORG_precision,▁
eval/combined_ORG_recall,▁
eval/combined_PER_f1,▁
+52,...


## Inspecting testing results on gold-only datasets

In [35]:
# Most important metrics
print(f"Overall F1 score:               {test_results["eval_combined_overall_f1"]*100:.2f}%")
print(f"F1 score on Hungarian dataset:  {test_results["eval_test_hun_overall_f1"]*100:.2f}%")
print(f"F1 score on German dataset:     {test_results["eval_test_ger_overall_f1"]*100:.2f}%")

Overall F1 score:               89.75%
F1 score on Hungarian dataset:  96.47%
F1 score on German dataset:     85.64%


Labels per category in the Hungarian dataset

In [37]:
# Hungarian
print(f"F1 score on PER labels:     {test_results["eval_test_hun_PER_f1"]*100:.2f}%")
print(f"F1 score on LOC labels:     {test_results["eval_test_hun_LOC_f1"]*100:.2f}%")
print(f"F1 score on ORG labels:     {test_results["eval_test_hun_ORG_f1"]*100:.2f}%")
print(f"F1 score on MISC labels:    {test_results["eval_test_hun_MISC_f1"]*100:.2f}%")

F1 score on PER labels:     96.55%
F1 score on LOC labels:     95.67%
F1 score on ORG labels:     98.40%
F1 score on MISC labels:    85.49%


Labels per category in the German dataset

In [38]:
# German
print(f"F1 score on PER labels:     {test_results["eval_test_ger_PER_f1"]*100:.2f}%")
print(f"F1 score on LOC labels:     {test_results["eval_test_ger_LOC_f1"]*100:.2f}%")
print(f"F1 score on ORG labels:     {test_results["eval_test_ger_ORG_f1"]*100:.2f}%")
print(f"F1 score on MISC labels:    {test_results["eval_test_ger_MISC_f1"]*100:.2f}%")

F1 score on PER labels:     90.33%
F1 score on LOC labels:     87.83%
F1 score on ORG labels:     76.83%
F1 score on MISC labels:    0.00%


By removing the English dataset we could prove that the model (xlm-roberta-base) is fully capable of learning the task, yielding ~96% on Hungarian and ~85% on German language.

If we include the English dataset, the end users will experience terrible performance on English text. This destroys trust in the model. So I would recommend sending the Hun-Ger NER model to production and getting a better quality English dataset to be able to get decent performance (85+%) on all languages.

## Saving the model and pushing to HuggingFace Hub

In [39]:
# Creating folder if it does not exist

os.makedirs("../models", exist_ok=True)

save_directory = "../models/xlm-roberta-ner-hun_ger"
# Saving the model
trainer.save_model(save_directory)

# Saving the tokenizer
tokenizer.save_pretrained(save_directory)

print(f"Model and tokenizer successfully saved to {save_directory}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer successfully saved to ../models/xlm-roberta-ner-hun_ger


In [41]:
# Pushing model to HuggingFace Hub
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/RichardHolzhofer/xlm-roberta-ner-hun_ger/commit/c16c42f2557d81099ad835a3ea913ac6b0d7f69e', commit_message='End of training', commit_description='', oid='c16c42f2557d81099ad835a3ea913ac6b0d7f69e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/RichardHolzhofer/xlm-roberta-ner-hun_ger', endpoint='https://huggingface.co', repo_type='model', repo_id='RichardHolzhofer/xlm-roberta-ner-hun_ger'), pr_revision=None, pr_num=None)

## Inferencing with the model via HuggingFace Hub

In [45]:
ner_pipeline = pipeline(
    "ner",
    model="RichardHolzhofer/xlm-roberta-ner-hun_ger",
    aggregation_strategy="simple"
    )

# Hungarian Example
# Translation: "János Kovács visited the OTP Bank headquarters in Budapest yesterday."
hun_text = "Kovács János tegnap az OTP Bank központjában, Budapesten járt."

print("--- Hungarian Prediction ---")
for entity in ner_pipeline(hun_text):
    print(f"Word: {entity['word']:<15} | Label: {entity['entity_group']:<5} | Score: {entity['score']:.4f}")
print("\n" + "="*50 + "\n")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- Hungarian Prediction ---
Word: Kovács János    | Label: PER   | Score: 0.9959
Word: OTP Bank        | Label: ORG   | Score: 0.9765
Word: Budapesten      | Label: LOC   | Score: 0.9962




In [46]:
# German Example
# Translation: "Angela Merkel visited the BMW headquarters in Munich yesterday."
ger_text = "Angela Merkel besuchte gestern die Zentrale von BMW in München."
print("--- German Prediction ---")
for entity in ner_pipeline(ger_text):
    print(f"Word: {entity['word']:<15} | Label: {entity['entity_group']:<5} | Score: {entity['score']:.4f}")

--- German Prediction ---
Word: Angela Merkel   | Label: PER   | Score: 0.9913
Word: BMW             | Label: ORG   | Score: 0.9971
Word: München         | Label: LOC   | Score: 0.9968
